In [ ]:
import pandas as pd
from datetime import datetime

data = pd.read_excel("data.xlsx")

data = data.drop_duplicates()

columns = []

for single_data in data :
    if single_data == "":
        single_data = pd.isna()
    if data[single_data][0] == pd.StringDtype:
        columns.append(single_data)

for col in columns:
    data[col] = data[col].str.strip()

numerical_vars = ["CustomerID", "Quantity", "UnitPrice"]

for col in numerical_vars:
    data[col] = data[col].astype(float)
    data[col] = data[col].fillna(0)

data["Description"] = data[col].fillna("No description")
data = data.drop(columns=["StockCode"])

print(data.head())
print(data.shape)
print(data.describe())
print(data.info())

  InvoiceNo  Description  Quantity         InvoiceDate  UnitPrice  CustomerID  \
0    536365         2.55       6.0 2010-12-01 08:26:00       2.55     17850.0   
1    536365         3.39       6.0 2010-12-01 08:26:00       3.39     17850.0   
2    536365         2.75       8.0 2010-12-01 08:26:00       2.75     17850.0   
3    536365         3.39       6.0 2010-12-01 08:26:00       3.39     17850.0   
4    536365         3.39       6.0 2010-12-01 08:26:00       3.39     17850.0   

          Country  
0  United Kingdom  
1  United Kingdom  
2  United Kingdom  
3  United Kingdom  
4  United Kingdom  
(536641, 7)
         Description       Quantity                 InvoiceDate  \
count  536641.000000  536641.000000                      536641   
mean        4.632656       9.620029  2011-07-04 08:57:06.087421   
min    -11062.060000  -80995.000000         2010-12-01 08:26:00   
25%         1.250000       1.000000         2011-03-28 10:52:00   
50%         2.080000       3.000000         20

In [9]:
data["Montant"] = data["Quantity"] * data["UnitPrice"]
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"], errors='coerce')

rfm = []

for customer_id, client_df in data.groupby("CustomerID"):
    rfm.append({
        "CustomerID": customer_id,
        "Recence": (pd.Timestamp.now() - client_df["InvoiceDate"].max()).days,
        "Frequence": client_df["InvoiceNo"].nunique(),
        "Montant": client_df["Montant"].sum()
    })

rfm = pd.DataFrame(rfm)
print(rfm.head().sort_values(by="Recence"))


   CustomerID  Recence  Frequence     Montant
0         0.0     5407       3710  1447487.53
2     12347.0     5409          7     4310.00
4     12349.0     5426          1     1757.55
3     12348.0     5482          4     1797.24
1     12346.0     5732          2        0.00


In [ ]:
import numpy as np
from sklearn.cluster import KMeans

encoded_data = pd.get_dummies(data=data, columns=["Country"], dtype=int)
featured_data = encoded_data.select_dtypes(include=[np.number])

kmeans = KMeans(
    n_clusters=2,
    init="k-means++",
    random_state=42,
    n_init=10
)

kmeans.fit(featured_data)

print(kmeans.predict(featured_data))
print(kmeans.labels_)
print(kmeans.cluster_centers_)

[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[[ 3.47406364e+00  1.21832726e+01  3.47406364e+00  1.52811608e+04
   2.06136379e+01  3.13243892e-03  9.98496031e-04  4.23302557e-05
   5.15184112e-03  7.96804813e-05  3.75992271e-04  1.88494139e-03
   1.52139919e-03  7.47004512e-05  9.68615850e-04  1.86128624e-02
   1.51890917e-04  1.73056045e-03  2.11028775e-02  2.36053426e-02
   3.63542196e-04 -3.59521440e-16  4.53182737e-04  6.15033715e-04
   1.99948208e-03  8.91425384e-04  1.12050677e-04  8.71505264e-05
   3.16231910e-04  5.90382566e-03  2.70415633e-03  8.49095129e-04
   3.66281212e-03  1.44420872e-04  2.49001504e-05  5.70213444e-04
   6.29475802e-03  1.14789693e-03  4.67375823e-03  7.24594377e-04
   1.69321023e-04  8.88258085e-01  6.00093625e-04]
 [ 8.07834238e+00  1.99686753e+00  8.07834238e+00  6.40284270e-10
   1.07191920e+01  1.82145965e-16 -7.68699340e-17  1.48107556e-05
  -5.44269491e-16 -1.77538106e-18 -4.14707331e-17 -1.56125113e-17
   1.04083409e-16  1.27597043e-17 -1.39211559e-16  5.25

class
Business    52540.081124
Economy      6572.342383
Name: price, dtype: float64
0         2743.317972
1         2554.935622
2         2744.700461
3         2646.666667
4         2555.793991
             ...     
300148    6871.527778
300149    7399.712092
300150    5719.378163
300151    8158.500000
300152    8093.750000
Length: 300153, dtype: float64
airline
AirAsia       4091.072742
Air_India    23507.019112
GO_FIRST      5652.007595
Indigo        5324.216303
SpiceJet      6179.278881
Vistara      30396.536302
Name: price, dtype: float64
stops
0    4499.0
1    7959.0
2    8307.0
Name: price, dtype: float64
days_left
1.0      41607528.0
2.0     121630693.0
3.0     123090403.0
4.0     130635808.0
5.0     143857338.0
6.0     142676275.0
7.0     145930459.0
8.0     143574563.0
9.0     145739184.0
10.0    148884953.0
11.0    147531040.0
12.0    143609531.0
13.0    144082862.0
14.0    143982637.0
15.0    139179109.0
16.0    128598242.0
17.0    130860006.0
18.0    131957113.0
19.0    127

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

categorical_cols = [
    "class",
    "airline",
    "source_city",
    "departure_time",
    "arrival_time",
    "destination_city",
]

numerical_cols = [
    "days_left",
    "duration"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

encoded_cat = pd.get_dummies(data, columns=categorical_cols, dtype=int)

final_features = encoded_cat.drop(columns=["price"])
target = encoded_cat["price"]


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor


x = final_features
y = target

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    train_size=0.8,
    random_state=42,
)

dummy_model = DummyRegressor(strategy="mean")

dummy_model.fit(x_train, y_train)

dummy_pred = dummy_model.predict(x_test)
dummy_mae = mean_absolute_error(y_test, dummy_pred)

print(dummy_mae)


19768.69424587037


In [ ]:
from sklearn.linear_model import LinearRegression


model = LinearRegression()

model.fit(x_train, y_train)

y_pred = model.predict(x_test)

linear_mae = mean_absolute_error(y_test, y_pred)
linear_mse = mean_squared_error(y_test, y_pred)
linear_rmse = linear_mse ** 0.5
linear_r2 = r2_score(y_test, y_pred)

print(linear_mse)
print(linear_rmse)
print(linear_r2)
print(linear_mae)



46443347.71222663
6814.935635222583
0.9099031138535913
4500.712501541964


4500.696062322511
46443340.39630758
6814.935098466278
0.9099031280459688


1073.4309735672211
7705004.546776118
2775.788995362601
0.9850528234590269


Scores for each fold: [0.98503368 0.98547687 0.98582626 0.98558446 0.98600483]
Average R² Score: 0.9856


Dummy (Baseline) -> Score R² sur le test: -0.0000
Linear Regression -> Score R² sur le test: 0.9099
Ridge Regression -> Score R² sur le test: 0.9099
Random Forest Regressor -> Score R² sur le test: 0.9849
